[Reference](https://github.com/Mega4alik/ollm)

In [1]:
!git clone https://github.com/Mega4alik/ollm.git

Cloning into 'ollm'...
remote: Enumerating objects: 477, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 477 (delta 73), reused 71 (delta 58), pack-reused 358 (from 1)
Receiving objects: 100% (477/477), 265.77 KiB | 1.48 MiB/s, done.
Resolving deltas: 100% (305/305), done.


In [2]:
%cd ollm

/content/ollm


In [3]:
!pip install -e .
!pip install kvikio-cu13

Obtaining file:///content/ollm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 101.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.6/274.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.6/413.6 kB 38.3 MB/s eta 0:00:00
  Building editable for ollm (pyproject.toml) ... done
  Created wheel for ollm: filename=ollm-0.5.2-0.editable-py3-none-any.whl size=5484 sha256=1c2fb141b0ee7766de1e8ce05fba6ee59bc8745645405144849be378226aa6d1
  Stored in directory: /tmp/pip-ephem-wheel-cache-ew1ggc4v/wheels/16/46/bc/4f78fb648747b4e526b3783c940e22477a1c371e3f5dfa0f7f
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=256040057 sha256=f25da18657a87fc83dc1bfb8b7751b82

In [2]:
from ollm import Inference, file_get_contents, TextStreamer

o = Inference("llama3-1B-chat", device="cuda:0", logging=True) #llama3-1B-chat(3B, 8B) | gpt-oss-20B
o.ini_model(models_dir="/media/mega4alik/ssd/models/", force_download=False)
o.offload_layers_to_cpu(layers_num=2) #offload some layers to CPU for speed increase
past_key_values = o.DiskCache(cache_dir="/media/mega4alik/ssd/kv_cache/")
text_streamer = TextStreamer(o.tokenizer, skip_prompt=True, skip_special_tokens=False)

sm, um = "You are helpful AI assistant", "List planets starting from Mercury"
#sm, um = file_get_contents("./samples/85k_sample.txt"), "What's common between these articles?"
messages = [{"role":"system", "content":sm}, {"role":"user", "content":um}]
input_ids = o.tokenizer.apply_chat_template(messages, reasoning_effort="minimal", tokenize=True, add_generation_prompt=True, return_tensors="pt").to(o.device)
outputs = o.model.generate(input_ids=input_ids,  past_key_values=past_key_values, max_new_tokens=100, streamer=text_streamer).cpu()
answer = o.tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=False)
print(answer)